In [17]:
import numpy as np
import difflib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
movies_data= pd.read_csv('movies_metadata.csv').head(10000)
movies_data.head(-5)

C:\Users\User\AppData\Local\Temp\ipykernel_20060\2877478369.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_data= pd.read_csv('movies_metadata.csv').head(10000)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9990,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10752, 'n...",NaN,31065,tt0401504,en,Ike: Countdown to D-Day,This is the story of the senior-level preparat...,6.988241,/a094KlACWR9GMYTdSfCt1HUQHoX.jpg,[],"[{'iso_3166_1': 'NZ', 'name': 'New Zealand'}, ...",2004-05-31,0.0,89.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Ike: Countdown to D-Day,False,5.7,10.0
9991,False,NaN,500000,"[{'id': 27, 'name': 'Horror'}]",NaN,22033,tt0065597,en,Cry of the Banshee,In seventeenth century England Lord Whitman wa...,1.848123,/lYKBgRpucgefRHmIzqlQxjSOp1K.jpg,[{'name': 'American International Pictures (AI...,"[{'iso_3166_1': 'GB', 'name': 'United Kingdom'}]",1970-07-22,0.0,91.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Hell hath no fury like a sorceress scorned!,Cry of the Banshee,False,5.5,12.0
9992,False,NaN,0,"[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...",NaN,37992,tt0036260,en,Phantom Lady,A mystery woman is a murder suspect's only ali...,2.014953,/pSPH9NlSVbXIfBA8DLRRET3YBBc.jpg,"[{'name': 'Universal Pictures', 'id': 33}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1944-01-28,0.0,87.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,IT'S UN

In [19]:
movies_data.shape

(10000, 24)

In [20]:
movies_data.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [21]:
movies_data.duplicated().sum()

np.int64(3)

In [22]:
movies_data.isnull().sum()

adult                       0
belongs_to_collection    8579
budget                      0
genres                      0
homepage                 9338
id                          0
imdb_id                     1
original_language           0
original_title              0
overview                   29
popularity                  0
poster_path                31
production_companies        0
production_countries        0
release_date                5
revenue                     0
runtime                     6
spoken_languages            0
status                      8
tagline                  3108
title                       0
video                       0
vote_average                0
vote_count                  0
dtype: int64

In [23]:
selected__features = ['title','genres','overview','tagline']
print(selected__features)

['title', 'genres', 'overview', 'tagline']


In [24]:
for feature in selected__features:
    movies_data[feature] = movies_data[feature].fillna('')

In [25]:
combined_features = (
    movies_data['genres']
    + ' '
    + movies_data['overview']
    + ' '
    + movies_data['tagline']
)

In [26]:
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 532152 stored elements and shape (10000, 33585)>
  Coords	Values
  (0, 14671)	0.059235364915901194
  (0, 81)	0.08830357150508503
  (0, 20384)	0.059223518571074864
  (0, 1586)	0.08962884165011877
  (0, 374)	0.03948105546035299
  (0, 6227)	0.03936139151390411
  (0, 26)	0.06935909854036375
  (0, 10895)	0.05596311925826129
  (0, 17305)	0.10451085337796931
  (0, 4600)	0.04376776547284691
  (0, 33076)	0.4202884946403637
  (0, 1532)	0.4050046217057128
  (0, 30601)	0.1578888206311972
  (0, 17720)	0.08468472465279436
  (0, 13461)	0.1261570559492439
  (0, 14946)	0.05510688960156531
  (0, 14068)	0.06811286249198226
  (0, 25637)	0.11154809405993678
  (0, 31593)	0.08479696851153275
  (0, 3461)	0.12392262434238908
  (0, 4211)	0.10562990318418629
  (0, 4594)	0.48144525311655967
  (0, 17575)	0.17693469400124395
  (0, 21306)	0.12815040801652186
  (0, 30030)	0.04343139842514077
  :	:
  (9998, 26626)	0.19656184514309458
  (9998, 18533)	0.19656

In [27]:
cosine_sim = cosine_similarity(feature_vectors)
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 532152 stored elements and shape (10000, 33585)>
  Coords	Values
  (0, 14671)	0.059235364915901194
  (0, 81)	0.08830357150508503
  (0, 20384)	0.059223518571074864
  (0, 1586)	0.08962884165011877
  (0, 374)	0.03948105546035299
  (0, 6227)	0.03936139151390411
  (0, 26)	0.06935909854036375
  (0, 10895)	0.05596311925826129
  (0, 17305)	0.10451085337796931
  (0, 4600)	0.04376776547284691
  (0, 33076)	0.4202884946403637
  (0, 1532)	0.4050046217057128
  (0, 30601)	0.1578888206311972
  (0, 17720)	0.08468472465279436
  (0, 13461)	0.1261570559492439
  (0, 14946)	0.05510688960156531
  (0, 14068)	0.06811286249198226
  (0, 25637)	0.11154809405993678
  (0, 31593)	0.08479696851153275
  (0, 3461)	0.12392262434238908
  (0, 4211)	0.10562990318418629
  (0, 4594)	0.48144525311655967
  (0, 17575)	0.17693469400124395
  (0, 21306)	0.12815040801652186
  (0, 30030)	0.04343139842514077
  :	:
  (9998, 26626)	0.19656184514309458
  (9998, 18533)	0.19656

In [41]:
movie_name = input('Enter your movie name :  ')
list_of_all_titles = movies_data['title'].tolist()

matching_titles = [
    title for title in list_of_all_titles if movie_name.lower() in title.lower()
]
if matching_titles:
    closest_match = matching_titles[0]
    print(f'match found containing keyword : {closest_match}')
else:
    find_close_match = difflib.get_close_matches(
    movie_name, list_of_all_titles, 
    n=1, cutoff=0.2
)
if find_close_match:
    closest_match = find_close_match[0]
    print(f'closest match found : {closest_match}')
else:
    print('no matching movie found! ')

movie_index = movies_data[movies_data.title == closest_match].index[0]
similarity_score = list(enumerate(cosine_sim[movie_index]))

sorted_similar_movies = sorted(
    similarity_score, key=lambda x:x[1], reverse=True
)
print('\nMovies suggested for you :\n')
i = 1
for movie in sorted_similar_movies:
    index = movie[0]
    title_from_index = movies_data.iloc[index]['title']
    if i < 11:
        print(f'{i}. {title_from_index}')
        i += 1

match found containing keyword : Cry of the Banshee
closest match found : Stolen Summer

Movies suggested for you :

1. Stolen Summer
2. Always
3. A Guy Named Joe
4. Blue Chips
5. Top Gun
6. Danny Deckchair
7. It Should Happen to You
8. Children's Island
9. Pete's Dragon
10. Last Action Hero
